# Literature Assignment

## Introduction

This notebook explores the Top 500 "Greatest" Novels dataset, compiled by OCLC based on library holdings data from over 16,000 member libraries worldwide. While the list claims to represent the world's most important novels, this analysis investigates a central question: **how dominant are English-language works and authors in a list that purports to be global?**

We focus on three dimensions of potential bias:
1. **Language** — What proportion of novels were originally written in English?
2. **Author Nationality** — Where do the authors come from?
3. **Gender** — How are male and female authors distributed across languages and rankings?

We intentionally limit our scope to these three dimensions to keep the analysis focused and readable. We do not analyze Goodreads ratings, OCLC holdings counts, or full-text content in this notebook.

**Dataset source:** [Responsible Datasets in Context — Top 500 "Greatest" Novels](https://www.responsible-datasets-in-context.com/posts/top-500-novels/top-500-novels.html)

In [2]:
import pandas as pd
import altair as alt

top_novels_url = "https://raw.githubusercontent.com/melaniewalsh/responsible-datasets-in-context/refs/heads/main/datasets/top-500-novels/top-500-novels-metadata_2025-01-11.csv"
top_novels_df = pd.read_csv(top_novels_url, low_memory=False)

print(f"Dataset shape: {top_novels_df.shape}")
print(f"Columns: {top_novels_df.columns.tolist()}")

Dataset shape: (500, 29)
Columns: ['top_500_rank', 'title', 'author', 'pub_year', 'orig_lang', 'genre', 'author_birth', 'author_death', 'author_gender', 'author_primary_lang', 'author_nationality', 'author_field_of_activity', 'author_occupation', 'oclc_holdings', 'oclc_eholdings', 'oclc_total_editions', 'oclc_holdings_rank', 'oclc_editions_rank', 'gr_avg_rating', 'gr_num_ratings', 'gr_num_reviews', 'gr_avg_rating_rank', 'gr_num_ratings_rank', 'oclc_owi', 'author_viaf', 'gr_url', 'wiki_url', 'pg_eng_url', 'pg_orig_url']


## Data Overview

Before diving into visualization, we examine the basic structure of the dataset 
and identify where data is missing — since gaps in the data are themselves 
meaningful and will affect our interpretations later.

In [8]:
print("Shape:", top_novels_df.shape)
print("\nData types:")
print(top_novels_df.dtypes)
print("\nFirst five rows:")
top_novels_df.head()

Shape: (500, 30)

Data types:
top_500_rank                  int64
title                        object
author                       object
pub_year                      int64
orig_lang                    object
genre                        object
author_birth                 object
author_death                 object
author_gender                object
author_primary_lang          object
author_nationality           object
author_field_of_activity     object
author_occupation            object
oclc_holdings               float64
oclc_eholdings              float64
oclc_total_editions         float64
oclc_holdings_rank          float64
oclc_editions_rank          float64
gr_avg_rating               float64
gr_num_ratings               object
gr_num_reviews               object
gr_avg_rating_rank            int64
gr_num_ratings_rank           int64
oclc_owi                    float64
author_viaf                 float64
gr_url                       object
wiki_url                     objec

,top_500_rank,title,author,pub_year,orig_lang,genre,author_birth,author_death,author_gender,author_primary_lang,...,gr_num_reviews,gr_avg_rating_rank,gr_num_ratings_rank,oclc_owi,author_viaf,gr_url,wiki_url,pg_eng_url,pg_orig_url,lang_group
0,1,Don Quixote,Miguel de Cervantes,1605,Spanish,action,1547,1616,male,spa,...,"12,053",318,211,1.810748e+09,17220427.0,https://www.goodreads.com/book/show/3836.Don_Q...,https://en.wikipedia.org/wiki/Don_Quixote,https://www.gutenberg.org/cache/epub/996/pg996...,https://www.gutenberg.org/cache/epub/2000/pg20...,Non-English
1,2,Alice's Adventures in Wonderland,Lewis Carroll,1865,English,fantasy,1832,1898,male,eng,...,"15,380",172,133,1.156132e+10,66462036.0,https://www.goodreads.com/book/show/24213.Alic...,https://en.wikipedia.org/wiki/Alice%27s_Advent...,https://www.gutenberg.org/cache/epub/11/pg11.txt,NaN,English
2,3,The Adventures of Huckleberry Finn,Mark Twain,1884,English,action,1835,1910,male,eng,...,"19,440",373,68,3.373178e+09,50566653.0,https://www.goodreads.com/book/show/2956.The_A...,https://en.wikipedia.org/wiki/Adventures_of_Hu...,https://www.gutenberg.org/cache/epub/76/pg76.txt,NaN,English
3,4,The Adventures of Tom Sawyer,Mark Twain,1876,English,action,1835,1910,male,eng,...,"13,603",301,88,3.373178e+09,50566653.0,https://www.goodreads.com/book/show/24583.The_...,https://en.wikipedia.org/wiki/The_Adventures_o...,https://www.gutenberg.org/cache/epub/74/pg74.txt,NaN,English
4,5,Treasure Island,Robert Louis Stevenson,1883,English,action,1850,1894,male,eng,...,"16,307",368,145,3.434000e+03,95207986.0,https://www.goodreads.com/book/show/295.Treasu...,https://en.wikipedia.org/wiki/Treasure_Island,https://www.gutenberg.org/cache/epub/120/pg120...,NaN,English


In [9]:
# Missing values
missing = top_novels_df.isna().sum()
missing_pct = (missing / len(top_novels_df) * 100).round(1)
missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_percent': missing_pct
}).query('missing_count > 0').sort_values('missing_count', ascending=False)

missing_df

,missing_count,missing_percent
pg_orig_url,436,87.2
author_field_of_activity,171,34.2
author_occupation,41,8.2
gr_url,20,4.0
oclc_holdings,5,1.0
oclc_eholdings,5,1.0
oclc_total_editions,5,1.0
oclc_holdings_rank,5,1.0
oclc_editions_rank,5,1.0
oclc_owi,5,1.0


In [10]:
# Data Cleaning: create binary language group column used in Section 3
top_novels_df['lang_group'] = top_novels_df['orig_lang'].apply(
    lambda x: 'English' if x == 'English' else 'Non-English'
)

# Check value counts for key columns we will analyze
print("orig_lang value counts (top 10):")
print(top_novels_df['orig_lang'].value_counts().head(10))

print("\nauthor_gender value counts:")
print(top_novels_df['author_gender'].value_counts(dropna=False))

print("\nlang_group value counts:")
print(top_novels_df['lang_group'].value_counts())

orig_lang value counts (top 10):
orig_lang
English       430
French         25
German         14
Russian        11
Spanish         7
Italian         5
Swedish         3
Latin           1
Japanese        1
Portuguese      1
Name: count, dtype: int64

author_gender value counts:
author_gender
male      355
female    145
Name: count, dtype: int64

lang_group value counts:
lang_group
English        430
Non-English     70
Name: count, dtype: int64


## Section 1: Language Distribution — How Dominant is English?

We begin by examining the original language of each novel. Since this list is compiled largely from English-speaking library systems (OCLC member libraries are predominantly in the US, UK, and Canada), we expect English to be overrepresented relative to its share of global literature.

In [3]:
# Count novels by original language
lang_counts = top_novels_df['orig_lang'].value_counts().reset_index()
lang_counts.columns = ['language', 'count']

# Label languages with fewer than 10 novels as "Other"
lang_counts['language_grouped'] = lang_counts['language'].where(
    lang_counts['count'] >= 10, other='Other'
)
lang_grouped = lang_counts.groupby('language_grouped')['count'].sum().reset_index()
lang_grouped = lang_grouped.sort_values('count', ascending=False)

# Calculate percentages
total = lang_grouped['count'].sum()
lang_grouped['percentage'] = (lang_grouped['count'] / total * 100).round(1)
lang_grouped['label'] = lang_grouped['percentage'].astype(str) + '%'

chart_lang = alt.Chart(lang_grouped).mark_bar().encode(
    x=alt.X('count:Q', title='Number of Novels'),
    y=alt.Y('language_grouped:N', sort='-x', title='Original Language'),
    color=alt.condition(
        alt.datum.language_grouped == 'English',
        alt.value('#e07b39'),
        alt.value('#4c78a8')
    ),
    tooltip=['language_grouped:N', 'count:Q', 'label:N']
).properties(
    title='Original Language of Top 500 Novels',
    width=500,
    height=300
)

text = chart_lang.mark_text(align='left', dx=3).encode(
    text='label:N'
)

(chart_lang + text)

alt.LayerChart(...)

**Findings:** 86% of the Top 500 novels were originally written in English — 
a striking figure given that English is the native language of roughly 5% of the 
world's population. French and German follow at a distant 5% and 2.8% respectively, 
while entire literary traditions — Arabic, Chinese, Hindi, Swahili, Bengali — 
are either absent or collapsed into the 4% "Other" category. This is not simply 
a reflection of what has been written; it reflects what Anglophone institutions 
have chosen to recognize, collect, and elevate.

## Section 2: Author Nationality — Whose Stories Are Centered?

Beyond language, we examine where authors are from. Even within English-language literature, some nationalities may be overrepresented. This section maps the geographic distribution of authorship in the Top 500.

In [7]:
# Count by nationality, group small ones
nat_counts = top_novels_df['author_nationality'].value_counts().reset_index()
nat_counts.columns = ['nationality', 'count']

nat_counts['nationality_grouped'] = nat_counts['nationality'].where(
    nat_counts['count'] >= 8, other='Other'
)
nat_grouped = nat_counts.groupby('nationality_grouped')['count'].sum().reset_index()
nat_grouped = nat_grouped.sort_values('count', ascending=False)

country_map = {
    'US': 'United States', 'GB': 'United Kingdom', 'FR': 'France',
    'DE': 'Germany', 'RU': 'Russia', 'CA': 'Canada', 'IE': 'Ireland',
    'Other': 'Other'
}
nat_grouped['nationality_grouped'] = nat_grouped['nationality_grouped'].map(
    lambda x: country_map.get(x, x)
)

chart_nat = alt.Chart(nat_grouped).mark_bar(color='#4c78a8').encode(
    x=alt.X('count:Q', title='Number of Authors'),
    y=alt.Y('nationality_grouped:N', sort='-x', title='Author Nationality'),
    tooltip=['nationality_grouped:N', 'count:Q']
).properties(
    title='Author Nationality in the Top 500 Novels',
    width=500,
    height=350
)

chart_nat

alt.Chart(...)

**Findings:** American and British authors together account for the majority of novels in the Top 500. This reinforces the pattern seen in the language analysis: the list is not simply biased toward English as a language, but specifically toward authors from the United States and United Kingdom. Authors from the Global South, East Asia, Africa, and the Middle East are strikingly underrepresented given the rich literary traditions in those regions.

## Section 3: Gender Distribution — Who Gets Canonized?

We now turn to author gender. Research consistently shows that female authors are underrepresented in literary canons. We examine how the gender gap varies across individual language traditions represented 
in the Top 500.

In [5]:
# Overall gender distribution
gender_counts = top_novels_df['author_gender'].value_counts().reset_index()
gender_counts.columns = ['gender', 'count']
total = gender_counts['count'].sum()
gender_counts['percentage'] = (gender_counts['count'] / total * 100).round(1)
gender_counts['label'] = gender_counts['percentage'].astype(str) + '%'

chart_gender = alt.Chart(gender_counts).mark_bar().encode(
    x=alt.X('gender:N', title='Author Gender'),
    y=alt.Y('count:Q', title='Number of Novels'),
    color=alt.Color('gender:N', scale=alt.Scale(
        domain=['male', 'female'],
        range=['#4c78a8', '#e07b39']
    )),
    tooltip=['gender:N', 'count:Q', 'label:N']
).properties(
    title='Author Gender Distribution in the Top 500 Novels',
    width=300,
    height=300
)

text_gender = chart_gender.mark_text(dy=-8, fontSize=13).encode(
    text='label:N'
)

(chart_gender + text_gender)

alt.LayerChart(...)

In [13]:
# Gender breakdown by language
gender_lang = top_novels_df.groupby(['orig_lang', 'author_gender']).size().reset_index(name='count')

# Calculate percentage within each language group
gender_lang['total'] = gender_lang.groupby('orig_lang')['count'].transform('sum')
gender_lang['percentage'] = (gender_lang['count'] / gender_lang['total'] * 100).round(1)

chart_gender_lang = alt.Chart(gender_lang).mark_bar().encode(
    x=alt.X('orig_lang:N', title='Language'),
    y=alt.Y('percentage:Q', title='Percentage of Novels (%)'),
    color=alt.Color('author_gender:N', scale=alt.Scale(
        domain=['male', 'female'],
        range=['#4c78a8', '#e07b39']
    ), legend=alt.Legend(title='Author Gender')),
    tooltip=['orig_lang:N', 'author_gender:N', 'count:Q', 'percentage:Q']
).properties(
    title='Gender Distribution by Original Language',
    width=550,
    height=300
)

chart_gender_lang

alt.Chart(...)

The gender gap varies significantly across languages. English sits at roughly 32% 
female authorship — already a minority, but notably higher than most other languages 
on the list. French and German come in lower at around 4% and 14% respectively, 
while Chinese, Italian, Latin, Polish, Portuguese, and Russian show no female authors 
at all in this dataset.

Two languages stand out as exceptions. Japanese novels in the Top 500 are entirely 
by female authors — a striking reversal that likely reflects the specific Japanese 
works selected (such as early court literature by authors like Murasaki Shikibu) 
rather than gender parity in Japanese publishing broadly. Spanish is the only 
language besides English where female authorship exceeds 40%, though the small 
number of Spanish novels on the list means this percentage is based on very few books.

The overall pattern reinforces the earlier finding: non-English literary traditions 
are not only underrepresented in volume, but when they do appear, they skew even 
more heavily male than English-language entries.

## Conclusions and Limitations

The data tells an 86% of the Top 500 novels were originally written in English, the language spoken natively by roughly 5% of the world's population. The authorship is even more concentrated than that: the United States alone accounts for more novels than all non-Anglophone countries combined, with the UK adding another large share. French, German, and Russian authors appear in small numbers; authors from Africa, South Asia, East Asia, and the Middle East are effectively invisible in this dataset.

The gender findings add another layer. Male authors make up 71% of the list overall, but the more striking number is in the non-English subset: only about 11% of non-English novels are by female authors, compared to 32% among English novels. This suggests that the barriers aren't simply additive — being a non-English-language female author doesn't just mean facing two separate disadvantages, but something closer to compounded exclusion from the canon entirely.

**Limitations:** What this dataset cannot tell us is whether these patterns reflect what has actually been written and published globally, or what Anglophone libraries have chosen to  collect. Probably both. OCLC's member libraries are concentrated in the US, UK, Canada, and Australia, so the list inevitably reflects their collecting priorities — which are themselves shaped by decades of publishing industry bias, translation markets, and institutional definitions of literary value.

Two concrete limitations are worth noting: First, the `author_nationality` and `author_gender` fields have missing values, so our counts slightly underrepresent the full picture. Second, collapsing languages below 10 novels into "Other" means we lose visibility into individual traditions — Arabic, Chinese, and Hindi literature, for instance, each likely have more than a handful of entries that disappear into that catch-all category.